# WBGT Learning Module 4
## Grid-cell labor-capacity loss: the input to the GTAP shock

This notebook finishes the **heat-stress → labor-capacity** chain of Saeed et al. (2022)
and produces the quantity that feeds their Equation 4, namely the per-grid-cell
labor-capacity loss by work intensity and indoor/outdoor condition,
\( z^{i,c}_{r,g} \).

### Where this sits in Saeed's Figure 1

- **Step A** (met → WBGT): Notebooks 1–2. Done.
- **Step B** (WBGT → grid-cell capacity via ISO 7243): completed here for **both**
  indoor and outdoor conditions and all four work intensities.
- **The +3 °C shock**: constructed here as a bias-corrected anomaly added to the
  ERA5 baseline (Saeed §2.3).
- **Step C** (grid → country × sector × labor type, Equation 4) and the **GTAP run**:
  **downstream, not in this notebook.** This notebook stops at the gridded
  \( z^{i,c}_{r,g} \) product and documents exactly what the next stage must add.


## 1. What this notebook needs as input

| Input | Variable(s) | Where it comes from |
|---|---|---|
| Baseline outdoor WBGT (subdaily) | `wbgt_outdoor_c` (°C) | your ERA5 baseline WBGT product |
| Baseline indoor WBGT (subdaily) | `wbgt_indoor_c` (°C) | same job, radiation set to 0 and wind fixed at 1 m/s |
| Daytime indicator | `rsds` (W m⁻²) | carried from the WBGT job |
| +3 °C anomaly | `dwbgt_outdoor_c`, `dwbgt_indoor_c` (°C) | CMIP6 SSP585 (+3 °C window) − CMIP6 historical (1961–1990), ensemble mean, regridded |

Two requirements from the Saeed methodology:

1. **Indoor WBGT must be computed separately** (shortwave = 0, wind = 1 m/s); it is
   *not* recoverable from outdoor WBGT. If `wbgt_indoor_c` is absent, this notebook
   runs outdoor-only and says so.
2. **Baseline and anomaly must share the same baseline period.** Saeed uses ERA5
   1961–1990. If your baseline is a different period (e.g. 1979–2008), compute the
   CMIP6 anomaly against that same period.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

# ---- Inputs -----------------------------------------------------------
# Baseline gridded WBGT product from Notebook 2 (has wbgt_outdoor_c, wbgt_indoor_c, rsds).
# Continent file:      wbgt_outputs/era5_africa_wbgt_baseline_week.nc
# Per-country crop:    wbgt_outputs/country/<country>_wbgt_baseline.nc  <-- EDIT
BASELINE_WBGT_FILE = Path("wbgt_outputs/era5_africa_wbgt_baseline_week.nc")

# +3 C bias-corrected WBGT anomaly.
#   "file"      : load ANOMALY_FILE (real CMIP6-derived product) <-- default for this course
#   "synthetic" : build a spatially varying stand-in so the notebook runs with no data
# The instructor bundles a real dwbgt_3c.nc (CMIP6 SSP585 +3 C window minus historical
# 1961-1990, ensemble mean, regridded to the baseline grid), so "file" is the default.
# Fall back to "synthetic" only if you have no anomaly file.
ANOMALY_MODE = "file"
ANOMALY_FILE = Path("wbgt_outputs/dwbgt_3c.nc")

# ---- Saeed / ISO 7243 parameters -------------------------------------
METABOLIC_RATES_W = [200.0, 300.0, 400.0, 600.0]   # light -> very heavy
RESTING_RATE_W = 117.0
DAYLIGHT_THRESHOLD_WM2 = 5.0                        # daytime working hours only

# ---- Optional country crop  -------
CROP_TO_COUNTRY = False        # needs regionmask + one-time Natural Earth download
COUNTRY_NAME = "Ghana"

OUTPUT_DIR = Path("gtap_shock_inputs")

## 2. Load and standardize the baseline WBGT product

In [2]:
def standardize(ds):
    """Rename coords/vars to the names this notebook uses."""
    ren = {}
    if "latitude" in ds.coords: ren["latitude"] = "lat"
    if "longitude" in ds.coords: ren["longitude"] = "lon"
    if "valid_time" in ds.coords: ren["valid_time"] = "time"
    ds = ds.rename(ren)
    # Outdoor WBGT under either name.
    if "wbgt_outdoor_c" not in ds:
        for alt in ("wbgt_baseline_c", "wbgt_c", "wbgt"):
            if alt in ds:
                ds = ds.rename({alt: "wbgt_outdoor_c"})
                break
    if "time" in ds.dims:
        ds = ds.transpose("time", "lat", "lon")
    return ds


baseline = standardize(xr.open_dataset(BASELINE_WBGT_FILE))

if "wbgt_outdoor_c" not in baseline:
    raise KeyError("Need an outdoor WBGT field (wbgt_outdoor_c / wbgt_baseline_c).")

HAS_INDOOR = "wbgt_indoor_c" in baseline
if not HAS_INDOOR:
    print("NOTE: no wbgt_indoor_c found - running OUTDOOR ONLY.")
    print("      For a full Saeed Step C, have the WBGT job also emit indoor WBGT")
    print("      (shortwave = 0, wind = 1 m/s).")

if "rsds" not in baseline:
    raise KeyError("Need rsds (W m-2) to define daytime working hours.")

CONDITIONS = ["outdoor"] + (["indoor"] if HAS_INDOOR else [])
print("Conditions:", CONDITIONS)
baseline

Conditions: ['outdoor', 'indoor']


<xarray.Dataset> Size: 12MB
Dimensions:         (time: 168, lat: 76, lon: 76)
Coordinates:
  * time            (time) datetime64[ns] 1kB 2020-07-01 ... 2020-07-07T23:00:00
  * lat             (lat) float64 608B -37.0 -36.0 -35.0 ... 36.0 37.0 38.0
  * lon             (lon) float64 608B -20.0 -19.0 -18.0 ... 53.0 54.0 55.0
Data variables:
    wbgt_outdoor_c  (time, lat, lon) float32 4MB ...
    wbgt_indoor_c   (time, lat, lon) float32 4MB ...
    rsds            (time, lat, lon) float32 4MB ...
Attributes:
    method:     Pure-Python explicit Liljegren WBGT (outdoor + indoor)
    data_mode:  aws
    period:     2020-07-01T00:00 to 2020-07-07T23:00 (hourly)
    note:       Baseline WBGT product for Notebook 4. Indoor: shortwave=0, wi...

## 3. Build the +3 °C shock (bias-corrected anomaly)

Saeed's +3 °C field is the ERA5 baseline **plus** a CMIP6-derived WBGT anomaly
(SSP585 years at 2.75–3.25 °C global warming, minus CMIP6 historical 1961–1990,
ensemble-mean, regridded). Adding the model *change* onto the observational
baseline is the bias correction — it avoids using CMIP6 absolute WBGT.

Here the anomaly is a **plug-in**: supply the real field via `ANOMALY_MODE="file"`,
or use the synthetic stand-in to exercise the pipeline. The synthetic anomaly is
spatially varying (larger inland/at lower latitude) so the resulting loss maps are
not uniform — but it is **not** a real climate signal.

In [3]:
def load_or_make_anomaly(baseline):
    """Return dwbgt_outdoor_c and dwbgt_indoor_c (deg C) on the baseline grid."""
    grid2d = baseline["wbgt_outdoor_c"].isel(time=0, drop=True)  # (lat, lon) template

    if ANOMALY_MODE == "file":
        dwb = standardize(xr.open_dataset(ANOMALY_FILE))
        d_out = dwb["dwbgt_outdoor_c"]
        if "dwbgt_indoor_c" in dwb:
            d_in = dwb["dwbgt_indoor_c"]
        else:
            print("WARNING: no dwbgt_indoor_c in anomaly file; reusing the outdoor "
                  "anomaly for indoor. Saeed computes indoor separately (it warms "
                  "less), so indoor loss will be biased high. Ship both fields.")
            d_in = d_out
        # align onto the baseline grid
        d_out = d_out.reindex_like(grid2d, method="nearest")
        d_in = d_in.reindex_like(grid2d, method="nearest")
        return d_out, d_in

    if ANOMALY_MODE == "synthetic":
        lat = baseline["lat"]
        lon = baseline["lon"]
        base = 2.4  # deg C, illustrative +3 C-world WBGT increase
        field = base + 0.9 * np.cos(np.deg2rad(lat)) + 0.006 * (lon - float(lon.mean()))
        d_out = (field.broadcast_like(grid2d)).rename("dwbgt_outdoor_c")
        d_out = d_out.transpose("lat", "lon")
        d_in = (0.85 * d_out).rename("dwbgt_indoor_c")  # indoor warms slightly less
        d_out.attrs["warning"] = "SYNTHETIC anomaly - not a real CMIP6 result"
        return d_out, d_in

    raise ValueError("ANOMALY_MODE must be 'file' or 'synthetic'.")


dwbgt_outdoor, dwbgt_indoor = load_or_make_anomaly(baseline)

anomaly = {"outdoor": dwbgt_outdoor, "indoor": dwbgt_indoor}

print("Mean +3C WBGT anomaly (deg C):")
for cond in CONDITIONS:
    print(f"  {cond:8s}: {float(anomaly[cond].mean()):.2f}")

plt.figure(figsize=(7, 4))
dwbgt_outdoor.plot(x="lon", y="lat")
plt.title("+3 C outdoor WBGT anomaly (plug-in field)")
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/junghawoo/Documents/GLASSNET_ECR/wbgt_outputs/dwbgt_3c.nc'

In [ ]:
# Future WBGT = baseline WBGT + anomaly, applied per condition, per timestep.
wbgt = {"baseline": {}, "future": {}}

wbgt["baseline"]["outdoor"] = baseline["wbgt_outdoor_c"]
wbgt["future"]["outdoor"] = baseline["wbgt_outdoor_c"] + anomaly["outdoor"]

if HAS_INDOOR:
    wbgt["baseline"]["indoor"] = baseline["wbgt_indoor_c"]
    wbgt["future"]["indoor"] = baseline["wbgt_indoor_c"] + anomaly["indoor"]

print("Future outdoor WBGT max (deg C):",
      round(float(wbgt["future"]["outdoor"].max()), 1))

## 4. ISO 7243 labor-response function (Saeed Eqs. 1–2)

\( WBGT_{lim}(M) = 56.7 - 11.5\log_{10}M \), rest limit at \(M = 117\,\mathrm{W}\),
and the clipped capacity ramp. Identical to the function used in Notebook 3 and in
`wbgt_physics.py`.

In [ ]:
def iso_wbgt_limit_c(metabolic_rate_w):
    return 56.7 - 11.5 * np.log10(metabolic_rate_w)


def iso_labor_capacity(wbgt_c, metabolic_rate_w):
    work_limit = iso_wbgt_limit_c(metabolic_rate_w)
    rest_limit = iso_wbgt_limit_c(RESTING_RATE_W)
    fraction = (rest_limit - wbgt_c) / (rest_limit - work_limit)
    return fraction.clip(0.0, 1.0)


pd.DataFrame({
    "Metabolic rate (W)": METABOLIC_RATES_W,
    "WBGT work limit (C)": [round(iso_wbgt_limit_c(m), 2) for m in METABOLIC_RATES_W],
})

## 5. Grid-cell capacity and capacity loss

Order matches Saeed: apply the nonlinear capacity function at every **daytime**
subdaily step, then average over time at each grid cell. The relative capacity loss

\[
z^{i,c}_{r,g} = \frac{\bar C^{\,c}_{\text{base}}(i) - \bar C^{\,c}_{\text{future}}(i)}
                     {\bar C^{\,c}_{\text{base}}(i)}
\]

is the per-cell, per-intensity, per-condition quantity that Equation 4 weights and
aggregates. We keep it as a **fraction in [0, 1]**; multiply by 100 for a percentage.

In [ ]:
daytime = baseline["rsds"] > DAYLIGHT_THRESHOLD_WM2

loss = xr.Dataset()          # gridded z^{i,c}_{r,g}
capacity_mean = xr.Dataset() # gridded baseline/future daytime-mean capacity

for cond in CONDITIONS:
    for m in METABOLIC_RATES_W:
        tag = f"{cond}_{int(m)}W"

        cap_base = iso_labor_capacity(wbgt["baseline"][cond], m).where(daytime).mean("time")
        cap_fut = iso_labor_capacity(wbgt["future"][cond], m).where(daytime).mean("time")

        z = ((cap_base - cap_fut) / cap_base).where(cap_base > 0.0)

        capacity_mean[f"capacity_baseline_{tag}"] = cap_base
        capacity_mean[f"capacity_future_{tag}"] = cap_fut
        loss[f"capacity_loss_{tag}"] = z

loss.attrs.update({
    "definition": "z = (baseline_capacity - future_capacity) / baseline_capacity, daytime-mean, fraction",
    "shock": "baseline -> +3 C, bias-corrected WBGT anomaly (Saeed 2022)",
    "conditions": ",".join(CONDITIONS),
    "note": "Grid-cell input to Saeed Eq. 4; NOT yet weighted by BLS shares or aggregated.",
})
loss

In [ ]:
# Quick look: outdoor relative loss at 400 W (heavy work).
example = "capacity_loss_outdoor_400W"
plt.figure(figsize=(7, 4))
(100.0 * loss[example]).plot(x="lon", y="lat")
plt.title("Relative labor-capacity loss (%), outdoor 400 W, +3 C")
plt.show()

rows = []
for c in CONDITIONS:
    for m in METABOLIC_RATES_W:
        z = loss[f"capacity_loss_{c}_{int(m)}W"]
        rows.append({"Condition x intensity": f"{c} {int(m)}W",
                     "Mean grid-cell loss (%)": round(float((100.0 * z).mean()), 2)})
pd.DataFrame(rows)

## 6. Crop to one country

Choose one country and carries it into their own GTAP
aggregation. This mask is the same polygon approach as Notebook 3 (centers inside
the national boundary). It needs `regionmask` and a one-time Natural Earth
download, so it is off by default.

In [ ]:
if CROP_TO_COUNTRY:
    import regionmask

    for version in ("natural_earth_v5_1_2", "natural_earth_v5_0_0", "natural_earth_v4_1_0"):
        if hasattr(regionmask.defined_regions, version):
            countries = getattr(regionmask.defined_regions, version).countries_110
            break

    number = next(n for n, name in zip(countries.numbers, countries.names)
                  if name.casefold() == COUNTRY_NAME.casefold())
    inside = countries.mask(loss["lon"], loss["lat"], wrap_lon=False) == number
    if int(inside.sum()) == 0:
        raise ValueError(f"No grid-cell centers inside {COUNTRY_NAME}; use a finer grid.")

    loss = loss.where(inside)
    capacity_mean = capacity_mean.where(inside)
    print(f"{COUNTRY_NAME}: {int(inside.sum())} grid cells retained.")
else:
    print("CROP_TO_COUNTRY is False - keeping the full grid.")

## 7. Save the hand-off product

This file contains the gridded
\( z^{i,c}_{r,g} \) (and the underlying capacities) for every intensity and
condition, ready for the next stage.

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)

handoff = xr.merge([loss, capacity_mean])
handoff.attrs.update({
    "product": "Grid-cell labor-capacity loss for Saeed Eq. 4 (GTAP shock input)",
    "shock": "baseline -> +3 C bias-corrected (CMIP6 anomaly on ERA5 baseline)",
    "anomaly_mode": ANOMALY_MODE,
    "conditions": ",".join(CONDITIONS),
    "intensities_W": ",".join(str(int(m)) for m in METABOLIC_RATES_W),
    "downstream_todo": (
        "Apply BLS ORS work-intensity (beta) and indoor/outdoor (alpha) shares; "
        "GDP-per-capita outdoor adjustment; aggregate grid->country with population "
        "(non-ag) and ag-production (ag) weights; map to GTAP v10 labor types; "
        "deliver as labor-augmenting productivity shock in the chosen closure."
    ),
})

# Name the output after the country so pre-cropped per-country files stay identifiable
# even when CROP_TO_COUNTRY is False (the file was already cropped upstream).
country_tag = COUNTRY_NAME.lower().replace(" ", "_").replace("'", "")
tag = country_tag if COUNTRY_NAME else "grid"
if not CROP_TO_COUNTRY:
    tag = f"{country_tag}_fullbox"   # data spans the file's bounding box, not the polygon
out_path = OUTPUT_DIR / f"capacity_loss_{tag}_3C.nc"
handoff.to_netcdf(out_path)
print("Saved:", out_path.resolve())

## 8. What the downstream stage must add (Saeed Equation 4 → GTAP)

This notebook produced \( z^{i,c}_{r,g} \). To reach the GTAP shock, the next stage
applies

\[
z_{l,a,r} = \sum_c \sum_i z^{i,c}_{r,g}\,\alpha^c_{l,a}\,\beta^i_{l,a},
\qquad
z_{l,r} = \text{aggregate}_g\big(z_{l,a,r}\big),
\]

which requires, in order:

1. **BLS ORS 2020 shares** — \( \alpha^c_{l,a} \) (indoor/outdoor exposure) and
   \( \beta^i_{l,a} \) (work-intensity mix) by labor type and activity.
2. **GDP-per-capita outdoor adjustment** — Saeed's exponential-decay downscaling of
   US outdoor-exposure shares to the country's income level.
3. **Aggregation weights** — gridded **population** (CIESIN GPW) for nonagricultural
   activities; gridded **agricultural production** (SIMPLE-G) for agricultural
   activities. (Area weights are *not* a substitute.)
4. **GTAP v10 concordance** — map to the region's labor types
   (skilled/unskilled × ag/non-ag) and deliver as a labor-augmenting productivity
   shock consistent with the chosen GEMPACK closure.

## References
- Saeed, W., et al. (2022). *The Poverty Impacts of Labor Heat Stress in West Africa Under a Warming Climate*. Earth's Future, 10, e2022EF002777.
- Kong, Q., & Huber, M. (2022). Earth's Future, 10, e2021EF002334.
- ISO 7243 (2017). *Hot environments — WBGT index*.